In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TradeCorp_Nettoyage") \
    .getOrCreate()

chemin = "/home/jovyan/data/"

df_categories = spark.read.csv(chemin + "categories.csv",header=True, inferSchema=True)
df_customers = spark.read.csv(chemin + "customers.csv",header=True, inferSchema=True)
df_employees = spark.read.csv(chemin + "employees.csv",header=True, inferSchema=True)
df_order_details = spark.read.csv(chemin + "order_details.csv",header=True, inferSchema=True)
df_orders = spark.read.csv(chemin + "orders.csv",header=True, inferSchema=True)
df_products = spark.read.csv(chemin + "products.csv",header=True, inferSchema=True)
df_shippers = spark.read.csv(chemin + "shippers.csv",header=True, inferSchema=True)
df_suppliers = spark.read.csv(chemin + "suppliers.csv",header=True, inferSchema=True)

print("---Spark est activé et toutes les tables sont chargées---")

---Spark est activé et toutes les tables sont chargées---


Q11—Valeurs nulles

In [2]:
from pyspark.sql.functions import col

print("---Comptage des valeurs nulles pour df_categories---")
for c in df_categories.columns:
    nb_null = df_categories.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")

print("---Comptage des valeurs nulles pour df_customers---")
for c in df_customers.columns:
    nb_null = df_customers.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")

print("---Comptage des valeurs nulles pour df_employees---")
for c in df_employees.columns:
    nb_null = df_employees.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")

print("---Comptage des valeurs nulles pour df_order_details---")
for c in df_order_details.columns:
    nb_null = df_order_details.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")

print("---Comptage des valeurs nulles pour df_orders---")
for c in df_orders.columns:
    nb_null = df_orders.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")

print("---Comptage des valeurs nulles pour df_products---")
for c in df_products.columns:
    nb_null = df_products.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")

print("---Comptage des valeurs nulles pour df_shippers---")
for c in df_shippers.columns:
    nb_null = df_shippers.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")

print("---Comptage des valeurs nulles pour df_suppliers---")
for c in df_suppliers.columns:
    nb_null = df_suppliers.filter(col(c).isNull()).count()
    print(f"{c}:{nb_null}")


---Comptage des valeurs nulles pour df_categories---
category_id:0
category_name:0
description:0
picture:8
---Comptage des valeurs nulles pour df_customers---
customer_id:0
company_name:0
contact_name:0
contact_title:0
address:0
city:0
region:60
postal_code:1
country:0
phone:0
fax:22
---Comptage des valeurs nulles pour df_employees---
employee_id:0
last_name:0
first_name:0
title:0
title_of_courtesy:0
birth_date:0
hire_date:0
address:0
city:0
region:4
postal_code:0
country:0
home_phone:0
extension:0
photo:9
notes:0
reports_to:1
photo_path:0
---Comptage des valeurs nulles pour df_order_details---
order_id:0
product_id:0
unit_price:0
quantity:0
discount:0
---Comptage des valeurs nulles pour df_orders---
order_id:0
customer_id:0
employee_id:0
order_date:0
required_date:0
shipped_date:21
ship_via:0
freight:0
ship_name:0
ship_address:0
ship_city:0
ship_region:507
ship_postal_code:19
ship_country:0
---Comptage des valeurs nulles pour df_products---
product_id:0
product_name:0
supplier_id:0
ca

Q12—Supprimer les nulls

In [3]:
# Suppression des lignes où shipped_date est null
df_orders = df_orders.dropna(subset=["shipped_date"])
print(f"Les lignes restantes dans df_orders : {df_orders.count()}")

# Remplacement dans df_products des valeurs nulles de unit_price par la médiane
from pyspark.sql.functions import median
# calcul de la médiane 
mediane_prix = df_products.select(median("unit_price")).first()[0]
print(f"Prix median calculé :{mediane_prix}")

# Remplacement des valeurs nulles 
df_products = df_products.na.fill(value=mediane_prix, subset=["unit_price"])



Les lignes restantes dans df_orders : 809
Prix median calculé :19.5


Q13—Cast de stypes

In [4]:
# conversion des dates dans df_orders
from pyspark.sql.types import DateType, IntegerType, DoubleType

df_orders = (df_orders.withColumn("order_date",col("order_date").cast(DateType()))
            .withColumn("required_date",col("required_date").cast(DateType()))
            .withColumn("shipped_date",col("shipped_date").cast(DateType())))

print("---Nouveau schéma de df_orders---")
df_orders.printSchema()

# conversion des prix et quantité dans df_order_details
df_order_details = (df_order_details.withColumn("unit_price",col("unit_price").cast(DoubleType()))
            .withColumn("quantity",col("quantity").cast(IntegerType())))

print("---Nouveau schéma de df_order_details---")
df_order_details.printSchema()           




---Nouveau schéma de df_orders---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)

---Nouveau schéma de df_order_details---
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



Q14—Nettoyage des chaînes

In [5]:
from pyspark.sql.functions import col,trim,initcap,upper

# application de TRIM sur toutes les colonnes textes de df_customers
for nom_colonne, type_colonne in df_customers.dtypes:
    if type_colonne == "string":
        df_customers = df_customers.withColumn(nom_colonne,trim(col(nom_colonne)))

# mettre contact_name en title case 
df_customers = df_customers.withColumn("contact_name",initcap(col("contact_name")))

# mettre country en MAJ avec upper
df_customers = df_customers.withColumn("country",upper(col("country")))

# visualisation du résultat 
df_customers.select("contact_name","country").show(5)



+------------------+-------+
|      contact_name|country|
+------------------+-------+
|      Maria Anders|GERMANY|
|      Ana Trujillo| MEXICO|
|    Antonio Moreno| MEXICO|
|      Thomas Hardy|     UK|
|Christina Berglund| SWEDEN|
+------------------+-------+
only showing top 5 rows


Q15—Renommer les colonnes

In [6]:
# renommer unit_price en prix_unitaire, quantity en quantité dans df_order_details
df_order_details = df_order_details.withColumnRenamed("unit_price","prix_unitaire")
df_order_details = df_order_details.withColumnRenamed("quantity","quantite")

# renommer ship_via en shipper_id dans la colonne df_orders
df_orders = df_orders.withColumnRenamed("ship_via","shipper_id")

# visualisation du résultat
df_order_details.show(5)
df_orders.show(5)

+--------+----------+-------------+--------+--------+
|order_id|product_id|prix_unitaire|quantite|discount|
+--------+----------+-------------+--------+--------+
|   10248|        11|         14.0|      12|     0.0|
|   10248|        42|          9.8|      10|     0.0|
|   10248|        72|         34.8|       5|     0.0|
|   10249|        14|         18.6|       9|     0.0|
|   10249|        51|         42.4|      40|     0.0|
+--------+----------+-------------+--------+--------+
only showing top 5 rows
+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+--------------------+--------------+-----------+----------------+------------+
|order_id|customer_id|employee_id|order_date|required_date|shipped_date|shipper_id|freight|           ship_name|        ship_address|     ship_city|ship_region|ship_postal_code|ship_country|
+--------+-----------+-----------+----------+-------------+------------+----------+-------+-----------------

Q16—Colonnes calculées